# Gold Order Fact

This notebook builds the `fact_orders` Gold model from the cleaned Silver orders dataset.

**Grain:** One row per `order_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_ORDERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/orders"
)

GOLD_FACT_ORDERS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/fact_orders"
)

print(f"Silver source: {SILVER_ORDERS_PATH}")
print(f"Gold target: {GOLD_FACT_ORDERS_PATH}")

## 2. Read Silver orders

In [0]:
silver_orders_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDERS_PATH)
)

silver_order_count = silver_orders_df.count()

print(f"Silver order rows: {silver_order_count:,}")

display(silver_orders_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_delayed",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_orders_df.columns)

if missing_columns:
    raise ValueError(
        f"Silver orders is missing required columns: {sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Build order fact

In [0]:
fact_orders_df = (
    silver_orders_df
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_delayed",
        "_silver_processed_at",
    )
    .withColumn(
        "purchase_date_key",
        F.date_format(
            F.to_date("order_purchase_timestamp"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "approved_date_key",
        F.date_format(
            F.to_date("order_approved_at"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "delivered_carrier_date_key",
        F.date_format(
            F.to_date("order_delivered_carrier_date"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "delivered_customer_date_key",
        F.date_format(
            F.to_date("order_delivered_customer_date"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "estimated_delivery_date_key",
        F.date_format(
            F.to_date("order_estimated_delivery_date"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(fact_orders_df.limit(10))

## 5. Validate order fact

In [0]:
fact_order_count = fact_orders_df.count()

duplicate_order_count = (
    fact_orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_order_id_count = (
    fact_orders_df
    .filter(F.col("order_id").isNull())
    .count()
)

if fact_order_count == 0:
    raise ValueError("Order fact is empty.")

if fact_order_count != silver_order_count:
    raise ValueError(
        "Order fact row count does not match Silver orders. "
        f"Silver: {silver_order_count:,}, Gold: {fact_order_count:,}"
    )

if duplicate_order_count > 0:
    raise ValueError(
        f"Order fact contains {duplicate_order_count:,} duplicate order IDs."
    )

if null_order_id_count > 0:
    raise ValueError(
        f"Order fact contains {null_order_id_count:,} null order IDs."
    )

print(f"Order fact rows: {fact_order_count:,}")
print("Order fact grain validation passed.")

## 6. Write order fact to Gold

In [0]:
(
    fact_orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FACT_ORDERS_PATH)
)

print(f"Order facts written to: {GOLD_FACT_ORDERS_PATH}")

## 7. Validate Gold output

In [0]:
written_order_facts_df = (
    spark.read
    .format("delta")
    .load(GOLD_FACT_ORDERS_PATH)
)

written_order_count = written_order_facts_df.count()

if written_order_count != fact_order_count:
    raise ValueError(
        "Gold order fact write validation failed. "
        f"Expected: {fact_order_count:,}, "
        f"Written: {written_order_count:,}"
    )

print(f"Written order fact rows: {written_order_count:,}")
print("Gold order fact write validation passed.")

## 8. Inspect Gold order fact

In [0]:
written_order_facts_df.printSchema()

display(
    written_order_facts_df
    .orderBy("order_id")
    .limit(20)
)